# OutplayArena — Play Against a Live Agent

This notebook plays **Prisoner's Dilemma** against a live LLM agent. You take one seat in the browser; this notebook plays the other.

**Before running this:** in the OutplayArena UI, configure a game with Player A = *Interactive (Human Player)* and Player B = *Remote Agent (API)*, click **Start**, then open the **API Keys** panel on the Play tab and copy the session key it shows you.

The OutplayArena SDK offers you an easy-to-start toolkit where you can either use pre-built agents or design your own (multi-)agent systems to explore cooperative and competitive behavior of AI agents.
**Note:** You can also use the OutplayArena MCP server to connect your existing agents like OpenClaw, Nous Research Hermes, etc. 


In [1]:
%pip install -q rich
%pip install -e agent-sdk/

/home/herbert/.cache/uv/builds-v0/.tmpiMQcxL/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/herbert/.cache/uv/builds-v0/.tmpiMQcxL/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import base64
import os

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import box

from outplayarena_sdk import LLMConfig, PrisonersDilemmaAgent

console = Console()
console.rule("[bold cyan]OutplayArena[/bold cyan]")

────────────────────────────────────────────────── OutplayArena ───────────────────────────────────────────────────

## Step 1 — Paste your session key

The key encodes `session_id:player:signature`, so decoding it locally tells us which session and which seat (`A` or `B`) this notebook is playing — no separate lookup needed.


In [3]:
SESSION_KEY = input("Paste your OutplayArena session key: ").strip()

session_id, player, _ = (
    base64.urlsafe_b64decode(SESSION_KEY.removeprefix("nks_") + "==")
    .decode()
    .split(":")
)

console.print(
    Panel.fit(
        f"Session [bold]{session_id}[/bold]\nPlaying as [bold yellow]Player {player}[/bold yellow]",
        title="Connected",
        border_style="green",
    )
)

Paste your OutplayArena session key:  nks_NzFiMzU1ZjQtNjdjMy00OGZkLThjZTEtM2M1OTUzNWNlYWY2OkI6M2VmNDg5OGYxMTllMjQwMWVhOTBjNTU5M2QxMDI5MGZmMGFiOTcwYzdmODlhZmI1M2RhYTg3OTIwYmI4M2RkZA


╭───────────────── Connected ──────────────────╮
│ Session 71b355f4-67c3-48fd-8ce1-3c59535ceaf6 │
│ Playing as Player B                          │
╰──────────────────────────────────────────────╯

## Step 2 — Configure the LLM

Using [OpenCode Zen](https://opencode.ai/zen) as the OpenAI-compatible backend, model `deepseek-v4-pro`. The API key is read from the `OPENCODE_API_KEY` environment variable — set it in your shell before launching the notebook so nothing sensitive is ever typed on screen. You can swap out the LLM API endpoint for your favorite provider anytime.


In [4]:
ARENA_URL = os.environ.get("ARENA_URL", "https://arena.core-aix.org/api")
MODEL = "deepseek-v4-pro"
OPENCODE_BASE_URL = "https://opencode.ai/zen/v1"

api_key = os.environ.get("OPENCODE_API_KEY")
if not api_key:
    raise RuntimeError("Set your API key before running this cell.")

llm_config = LLMConfig(model=MODEL, api_key=api_key, base_url=OPENCODE_BASE_URL)
console.print(f"LLM configured: [bold]{MODEL}[/bold] via {llm_config.base_url}")

LLM configured: deepseek-v4-pro via https://opencode.ai/zen/v1

## Step 3 — A live-logging agent

`BaseAgent` exposes hooks that fire as the game progresses (`on_round_start`, `on_action_decision`, `on_round_end`, ...). Overriding a few of them turns the otherwise-silent `run_sync()` call into a live commentary of the match.


In [5]:
class LoggingPDAgent(PrisonersDilemmaAgent):
    def __init__(self, *args, console: Console, **kwargs):
        super().__init__(*args, **kwargs)
        self.console = console
        self._announced_round = 0

    def on_episode_start(self, session_id, seed):
        self.console.rule(f"[bold cyan]Game on[/bold cyan] — session {session_id}")

    def on_round_start(self, round_num, state):
        if round_num != self._announced_round:
            self._announced_round = round_num
            total = state.get("round_total", "?")
            self.console.print(
                f"[dim]Round {round_num}/{total} — waiting on both players...[/dim]"
            )

    def on_action_decision(self, action, reasoning):
        self.console.print(
            f"  [bold yellow]Agent[/bold yellow] plays [bold]{action}[/bold]"
        )
        reasoning = reasoning.strip()
        if reasoning and reasoning.lower() != action:
            self.console.print(f"  [dim]› {reasoning[:160]}[/dim]")

    def on_round_end(self, round_num, state):
        history = state.get("history", [])
        if history and history[-1]["round"] == round_num:
            entry = history[-1]
            scores = entry["total_scores"]
            self.console.print(
                f"  [green]Round {round_num} resolved[/green] — "
                f"A: {entry['actions']['A']}  B: {entry['actions']['B']}  "
                f"→ scores A={scores['A']} B={scores['B']}"
            )

    def on_episode_end(self, results):
        self.console.rule("[bold green]Game complete[/bold green]")

## Step 4 — Play!

Run this cell, then switch to the browser tab and make your moves. Logs stream into this cell as each round resolves; the cell finishes once the match is over.


In [6]:
agent = LoggingPDAgent(
    player=player,
    player_token=SESSION_KEY,
    session_id=session_id,
    arena_url=ARENA_URL,
    llm_config=llm_config,
    console=console,
)

# The SDK agent base class implements an agent loop and listens to server events.
results = agent.run_sync()

───────────────────────────── Game on — session 71b355f4-67c3-48fd-8ce1-3c59535ceaf6 ──────────────────────────────

Round 1/3 — waiting on both players...

Agent plays cooperate

› I've sent a cooperative message. Now let me open with cooperation to build trust.

Round 1 resolved — A: cooperate  B: cooperate  → scores A=3.0 B=3.0

...

Round 3/3 — waiting on both players...

Agent plays defect

› This is the final round. We've built a nice cooperative relationship, but with no future rounds, the temptation
to defect is strong. Let me think about what A m

Round 3 resolved — A: defect  B: defect  → scores A=7.0 B=7.0

────────────────────────────────────────────────── Game complete ──────────────────────────────────────────────────

## Results


In [7]:
table = Table(title="Round-by-round", box=box.ROUNDED)
table.add_column("Round", justify="right")
table.add_column("Player A")
table.add_column("Player B")
table.add_column("Score A", justify="right")
table.add_column("Score B", justify="right")

for entry in results["history"]:
    scores = entry["total_scores"]
    table.add_row(
        str(entry["round"]),
        entry["actions"]["A"],
        entry["actions"]["B"],
        f"{scores['A']:.1f}",
        f"{scores['B']:.1f}",
    )

console.print(table)

scores = results["total_scores"]
console.print(
    Panel.fit(
        f"[bold]{results['winner']}[/bold] wins — A: {scores['A']:.1f}  B: {scores['B']:.1f}",
        title="Final Result",
        border_style="gold1",
    )
)

                   Round-by-round                    
╭───────┬───────────┬───────────┬─────────┬─────────╮
│ Round │ Player A  │ Player B  │ Score A │ Score B │
├───────┼───────────┼───────────┼─────────┼─────────┤
│     1 │ cooperate │ cooperate │     3.0 │     3.0 │
│     2 │ cooperate │ cooperate │     6.0 │     6.0 │
│     3 │ defect    │ defect    │     7.0 │     7.0 │
╰───────┴───────────┴───────────┴─────────┴─────────╯

╭────── Final Result ───────╮
│ Tie wins — A: 7.0  B: 7.0 │
╰───────────────────────────╯